# Adaptive Risk Fusion Comparison


## 1) Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2) Install Dependencies


In [ ]:
!pip install -q scikit-learn matplotlib seaborn pandas numpy torch


## 3) Load Dataset


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
required = ['X_train.npy','X_test.npy','y_train.npy','y_test.npy']
print('Data path:', DATA_PATH)
print('Files:', os.listdir(DATA_PATH))
missing = [f for f in required if not os.path.exists(os.path.join(DATA_PATH,f))]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

X_train = np.load(os.path.join(DATA_PATH,'X_train.npy'))
X_test = np.load(os.path.join(DATA_PATH,'X_test.npy'))
y_train = np.load(os.path.join(DATA_PATH,'y_train.npy')).reshape(-1)
y_test = np.load(os.path.join(DATA_PATH,'y_test.npy')).reshape(-1)
if X_train.ndim==3 and X_train.shape[-1]==1: X_train=X_train[...,0]
if X_test.ndim==3 and X_test.shape[-1]==1: X_test=X_test[...,0]
print('X_train:',X_train.shape,'X_test:',X_test.shape)



## 4) Generate Signals and Targets


In [ ]:
RNG_SEED=42
np.random.seed(RNG_SEED)

def minmax(v):
    v=np.asarray(v,dtype=np.float32)
    lo,hi=float(v.min()),float(v.max())
    if np.isclose(lo,hi):
        return np.full_like(v,0.5)
    return (v-lo)/(hi-lo)

def simulate_signals(X,y,seed):
    rng=np.random.default_rng(seed)
    a=minmax(np.mean(np.abs(X),axis=1))
    s=minmax(np.std(X,axis=1))
    yn=minmax(y.astype(np.float32))
    atk=(y>0).astype(np.float32)
    c=np.clip(0.5*a+0.35*atk+0.15*yn+rng.normal(0,0.04,len(y)),0,1)
    m=np.clip(0.45*c+0.3*yn+0.25*a+rng.normal(0,0.05,len(y)),0,1)
    g=np.clip(0.4*c+0.35*s+0.25*yn+rng.normal(0,0.05,len(y)),0,1)
    f=np.clip(0.55*a+0.3*s+0.15*c+rng.normal(0,0.04,len(y)),0,1)
    return np.column_stack([c,m,g,f]).astype(np.float32)

def build_targets(sig,y,seed):
    rng=np.random.default_rng(seed+1)
    c,m,g,f=sig[:,0],sig[:,1],sig[:,2],sig[:,3]
    risk=np.clip(0.42*c+0.24*m+0.2*g+0.14*f+0.08*(y>0)+rng.normal(0,0.03,len(y)),0,1)
    out=np.zeros(len(y),dtype=np.int32)
    out[risk>0.4]=1
    out[risk>0.6]=2
    out[risk>0.8]=3
    return out

Xtr=simulate_signals(X_train,y_train,RNG_SEED)
Xte=simulate_signals(X_test,y_test,RNG_SEED+10)
ytr=build_targets(Xtr,y_train,RNG_SEED)
yte=build_targets(Xte,y_test,RNG_SEED+10)



## 5) Define Fusion Methods


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
import torch.nn as nn

def sev_from_risk(r):
    if r>0.8: return 3
    if r>0.6: return 2
    if r>0.4: return 1
    return 0

def weighted(x):
    c,m,g,f=x
    return 0.45*c+0.25*m+0.2*g+0.1*f

def dynamic(x):
    c,m,g,f=x
    w=(0.6,0.2,0.1,0.1) if c>0.9 else (0.35,0.3,0.25,0.1)
    return w[0]*c+w[1]*m+w[2]*g+w[3]*f

def fuzzy(x):
    c,m,g,f=x
    if c>0.8 and m>0.8: r=0.9
    elif c>0.8 and g>0.8: r=0.7
    elif 0.5<=c<=0.8: r=0.5
    else: r=0.25
    return float(np.clip(r+0.05*f,0,1))

class Attn(nn.Module):
    def __init__(self):
        super().__init__(); self.fc=nn.Linear(4,4)
    def forward(self,x):
        w=torch.softmax(self.fc(x),dim=-1)
        r=torch.sigmoid(torch.sum(w*x,dim=-1,keepdim=True))
        return r,w

lr=LogisticRegression(max_iter=1000,random_state=42).fit(Xtr,ytr)
rf=RandomForestClassifier(n_estimators=100,random_state=42).fit(Xtr,ytr)
attn=Attn(); opt=torch.optim.Adam(attn.parameters(),lr=0.01); loss_fn=nn.MSELoss()
Xt=torch.tensor(Xtr,dtype=torch.float32); yt=torch.tensor((ytr/3.0).reshape(-1,1),dtype=torch.float32)
for _ in range(100):
    opt.zero_grad(); pr,_=attn(Xt); loss=loss_fn(pr,yt); loss.backward(); opt.step()



## 6) Evaluate Methods


In [ ]:
methods=['Weighted','Dynamic','LogisticRegression','RandomForest','Fuzzy','Attention']
rows=[]
for m in methods:
    preds=[]
    for x in Xte:
        if m=='Weighted': risk=weighted(x)
        elif m=='Dynamic': risk=dynamic(x)
        elif m=='Fuzzy': risk=fuzzy(x)
        elif m=='LogisticRegression':
            p=lr.predict_proba([x])[0]; c=lr.classes_.astype(float); risk=float(np.dot(p,c)/max(c.max(),1.0))
        elif m=='RandomForest':
            p=rf.predict_proba([x])[0]; c=rf.classes_.astype(float); risk=float(np.dot(p,c)/max(c.max(),1.0))
        else:
            with torch.no_grad(): risk=float(attn(torch.tensor([x],dtype=torch.float32))[0].item())
        preds.append(sev_from_risk(float(np.clip(risk,0,1))))
    preds=np.array(preds)
    rows.append({'Method':m,'Accuracy':accuracy_score(yte,preds),'Precision':precision_score(yte,preds,average='macro',zero_division=0),'Recall':recall_score(yte,preds,average='macro',zero_division=0),'F1':f1_score(yte,preds,average='macro',zero_division=0)})
results=pd.DataFrame(rows).sort_values('F1',ascending=False).reset_index(drop=True)
results



## 7) Charts and Best Method


In [ ]:
results.plot(x='Method',y='Accuracy',kind='bar',figsize=(9,4),ylim=(0,1),title='Accuracy Comparison')
plt.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.show()
results.plot(x='Method',y='F1',kind='bar',figsize=(9,4),ylim=(0,1),title='F1 Comparison')
plt.grid(axis='y',alpha=0.3); plt.tight_layout(); plt.show()
best=results.iloc[0]
print('Best Fusion Method:',best['Method'])
print(best.to_string())

